# 01  Name Reference and Persona Enrichment

This consolidated notebook replaces the earlier 01A and 02A enhancement notebooks.

It validates the reviewed cleaning rules and frequency list, resolves rule cycles, builds a reusable name-reference lookup, and enriches the existing persona context with canonical name keys, rarity, and gender evidence.

It preserves source values and does not change candidate pairs, labels, clusters, or IDs.


**What this code does:** Imports libraries, defines project folders, and checks that the required reviewed reference files are available.


In [1]:
# Cell 1  Imports, project paths, inputs, and outputs

from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError(f"Could not locate the project root from: {Path.cwd()}")

DATA_DIR = PROJECT_ROOT / "data"
REVIEWED_DIR = DATA_DIR / "reviewed"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
REVIEW_DIR = PROJECT_ROOT / "outputs" / "review"

RULES_PATH = REVIEWED_DIR / "name_cleaning_rules.json"
FREQUENCY_PATH = REVIEWED_DIR / "name_frequency_analysis_v13.csv"
PERSONA_OVERRIDE_PATH = REVIEWED_DIR / "persona_name_overrides_v1.csv"
GENDER_OVERRIDE_PATH = REVIEWED_DIR / "persona_gender_overrides_v1.csv"
VARIANT_MAPPING_PATH = REVIEWED_DIR / "approved_name_variant_mappings_v1.csv"
CONTEXT_PATH = INTERIM_DIR / "persona_context.parquet"

LOOKUP_PATH = PROCESSED_DIR / "name_reference_lookup_v1.csv"
ENRICHED_CONTEXT_PATH = INTERIM_DIR / "persona_context_name_enriched_v1.parquet"

REFERENCE_VALIDATION_PATH = REPORT_DIR / "name_reference_validation_v1.csv"
CONFLICT_PATH = REPORT_DIR / "name_reference_rule_conflicts_v1.csv"
CYCLE_AUDIT_PATH = REPORT_DIR / "name_reference_cycle_resolution_v1.csv"
CONTEXT_VALIDATION_PATH = REPORT_DIR / "name_context_enrichment_validation_v1.csv"
COVERAGE_PATH = REPORT_DIR / "name_context_enrichment_coverage_v1.csv"
GENDER_REVIEW_PATH = REVIEW_DIR / "name_reference_gender_conflicts_v1.csv"
OVERRIDE_AUDIT_PATH = REPORT_DIR / "persona_name_override_audit_v1.csv"
GENDER_OVERRIDE_AUDIT_PATH = REPORT_DIR / "persona_gender_override_audit_v1.csv"
VARIANT_AUDIT_PATH = REPORT_DIR / "approved_name_variant_application_audit_v1.csv"
CS_RULE_AUDIT_PATH = REPORT_DIR / "professor_approved_spanish_cs_rule_audit_v1.csv"
PLACE_LINKAGE_AUDIT_PATH = REPORT_DIR / "place_linkage_normalization_audit_v1.csv"
ROLE_REMAP_AUDIT_PATH = REPORT_DIR / "marriage_godparent_role_remap_audit_v1.csv"

for directory in [INTERIM_DIR, PROCESSED_DIR, REPORT_DIR, REVIEW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [RULES_PATH, FREQUENCY_PATH, PERSONA_OVERRIDE_PATH, GENDER_OVERRIDE_PATH, VARIANT_MAPPING_PATH, CONTEXT_PATH]:
    assert required_path.exists(), f"Missing required input: {required_path}"

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\samav\OneDrive\Desktop\sundondo


**What this code does:** Defines reusable text-cleaning helpers. These create comparison keys and do not replace the original historical spellings.


In [2]:
# Cell 2 — Safe text normalization

def repair_mojibake(value):
    """
    Repairs common UTF-8 text that was mistakenly interpreted as Latin-1.
    Examples: 'JosÃ©' -> 'José', 'MuÃ±oz' -> 'Muñoz'.
    """
    if pd.isna(value):
        return ""

    text = str(value)

    if any(marker in text for marker in ["Ã", "Â", "â"]):
        try:
            return text.encode("latin1").decode("utf-8")
        except UnicodeError:
            return text

    return text


def normalize_name(value):
    """
    Produces a stable key for matching.
    It preserves original source values separately, so this is never used
    to replace historical spelling.
    """
    text = repair_mojibake(value).strip().lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Mn"
    )

    text = re.sub(r"[^a-zñ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


**What this code does:** Reads the name-cleaning rules and name-frequency table supplied and reviewed by the research team.


In [3]:
# Cell 3 — Load the teammate reference files

with RULES_PATH.open("r", encoding="utf-8") as file:
    rules = json.load(file)

frequency = pd.read_csv(FREQUENCY_PATH, dtype=str)

required_frequency_columns = {
    "name_type",
    "name",
    "gender",
    "freq",
    "categorize",
    "event_id",
}

missing_columns = required_frequency_columns - set(frequency.columns)
assert not missing_columns, (
    f"Frequency file is missing required columns: {sorted(missing_columns)}"
)

frequency["name_original"] = frequency["name"].fillna("")
frequency["name_key"] = frequency["name_original"].map(normalize_name)
frequency["freq"] = pd.to_numeric(frequency["freq"], errors="coerce")

frequency.head()


,name_type,name,gender,freq,categorize,event_id,name_original,name_key
0,first_name,maria,Female,1505,Most common,bautizo-16|bautizo-16|bautizo-17|bautizo-18|ba...,maria,maria
1,first_name,manuela,Female,866,Most common,bautizo-15|bautizo-42|bautizo-43|bautizo-46|ba...,manuela,manuela
2,first_name,juana,Female,851,Most common,bautizo-33|bautizo-38|bautizo-47|bautizo-62|ba...,juana,juana
3,first_name,francisca,Female,543,Most common,bautizo-4|bautizo-7|bautizo-39|bautizo-50|baut...,francisca,francisca
4,first_name,josefa,Female,462,Most common,bautizo-5|bautizo-45|bautizo-78|bautizo-84|bau...,josefa,josefa


**What this code does:** Extracts the reviewed spelling-to-canonical mappings into a consistent format for later checks.


In [4]:
# Cell 4 — Extract and normalize the team’s mapping rules

first_name_abbreviations = rules["name_rules"]["abbreviations"]
first_name_variants = rules["name_rules"]["spelling_variants"]
last_name_variants = rules["lastname_rules"]["spelling_variants"]

first_name_rules = {
    **first_name_abbreviations,
    **first_name_variants,
}

def prepare_mapping(mapping, name_type, rule_source):
    rows = []

    for original, canonical in mapping.items():
        rows.append(
            {
                "name_type": name_type,
                "original_name": original,
                "original_name_key": normalize_name(original),
                "direct_canonical_name": canonical,
                "direct_canonical_key": normalize_name(canonical),
                "rule_source": rule_source,
            }
        )

    return pd.DataFrame(rows)

first_name_mapping = prepare_mapping(
    first_name_rules,
    name_type="first_name",
    rule_source="name_cleaning_rules.json",
)

approved_variants = pd.read_csv(VARIANT_MAPPING_PATH, dtype=str).fillna("")
required_variant_columns = {"source_name_key", "canonical_name_key", "mapping_type", "decision_basis", "approved_by", "approved_at"}
missing_variant_columns = required_variant_columns - set(approved_variants.columns)
assert not missing_variant_columns, f"Variant mapping file is missing: {sorted(missing_variant_columns)}"
assert not approved_variants["source_name_key"].duplicated().any(), "Variant mapping file has duplicate source names."
assert approved_variants["source_name_key"].str.strip().ne("").all(), "Variant mapping contains blank source names."
assert approved_variants["canonical_name_key"].str.strip().ne("").all(), "Variant mapping contains blank canonical names."

approved_variant_mapping = pd.DataFrame({
    "name_type": "first_name",
    "original_name": approved_variants["source_name_key"],
    "original_name_key": approved_variants["source_name_key"].map(normalize_name),
    "direct_canonical_name": approved_variants["canonical_name_key"],
    "direct_canonical_key": approved_variants["canonical_name_key"].map(normalize_name),
    "rule_source": "approved_name_variant_mappings_v1.csv",
})
first_name_mapping = pd.concat([first_name_mapping, approved_variant_mapping], ignore_index=True)

last_name_mapping = prepare_mapping(
    last_name_variants,
    name_type="last_name",
    rule_source="name_cleaning_rules.json",
)

rule_mapping = pd.concat(
    [first_name_mapping, last_name_mapping],
    ignore_index=True,
)

rule_mapping.head()


,name_type,original_name,original_name_key,direct_canonical_name,direct_canonical_key,rule_source
0,first_name,ma,ma,maria,maria,name_cleaning_rules.json
1,first_name,ped,ped,pedro,pedro,name_cleaning_rules.json
2,first_name,ygn,ygn,ygnacio,ygnacio,name_cleaning_rules.json
3,first_name,yga,yga,ygnacia,ygnacia,name_cleaning_rules.json
4,first_name,jph,jph,joseph,joseph,name_cleaning_rules.json


**What this code does:** Follows multi-step name mappings and safely handles circular mappings so a canonical comparison key can be created.


In [5]:
# Cell 5 — Resolve multi-step mappings and safely collapse circular rules

def resolve_final_canonical(start_key, direct_map):
    """
    Resolves mappings such as:
    agabito -> agavito -> agapito

    Returns a final key, a status, and the mapping path.
    """
    visited = []
    current = start_key

    while current in direct_map:
        if current in visited:
            cycle_path = " -> ".join(visited + [current])
            return current, "cycle_detected", cycle_path

        visited.append(current)
        current = direct_map[current]

    return current, "resolved", " -> ".join(visited + [current])


# Frequency decides the canonical spelling when two rules point at each other.
# A higher observed frequency wins; alphabetical order is only a deterministic tie-breaker.
frequency_evidence = (
    frequency[frequency["name_type"].isin(["first_name", "last_name"])]
    .groupby(["name_type", "name_key"], dropna=False)["freq"]
    .max()
    .to_dict()
)

def find_cycle_nodes(start_key, direct_map):
    visited = []
    current = start_key

    while current in direct_map:
        if current in visited:
            return visited[visited.index(current):]
        visited.append(current)
        current = direct_map[current]

    return []


resolved_parts = []
cycle_audits = []

for name_type, group in rule_mapping.groupby("name_type", dropna=False):
    direct_map = (
        group.drop_duplicates("original_name_key")
        .set_index("original_name_key")["direct_canonical_key"]
        .to_dict()
    )

    # Collapse each cycle to one evidence-backed canonical spelling.
    unique_cycles = set()
    for key in direct_map:
        cycle_nodes = find_cycle_nodes(key, direct_map)
        if cycle_nodes:
            unique_cycles.add(tuple(sorted(cycle_nodes)))

    for cycle_nodes in sorted(unique_cycles):
        canonical_key = min(
            cycle_nodes,
            key=lambda key: (-frequency_evidence.get((name_type, key), 0), key),
        )

        for key in cycle_nodes:
            if key == canonical_key:
                direct_map.pop(key, None)
            else:
                direct_map[key] = canonical_key

        cycle_audits.append(
            {
                "name_type": name_type,
                "cycle_nodes": " | ".join(cycle_nodes),
                "chosen_canonical_key": canonical_key,
                "chosen_canonical_frequency": frequency_evidence.get((name_type, canonical_key), 0),
                "resolution_method": "highest_observed_frequency_then_alphabetical",
            }
        )

    group = group.copy()

    result = group["original_name_key"].map(
        lambda value: resolve_final_canonical(value, direct_map)
    )

    group["canonical_name_key"] = result.map(lambda item: item[0])
    group["mapping_status"] = result.map(lambda item: item[1])
    group["mapping_path"] = result.map(lambda item: item[2])

    resolved_parts.append(group)

resolved_rules = pd.concat(resolved_parts, ignore_index=True)

cycle_resolution_audit = pd.DataFrame(cycle_audits)
cycle_resolution_audit.to_csv(CYCLE_AUDIT_PATH, index=False)

residual_cycles = resolved_rules[
    resolved_rules["mapping_status"].eq("cycle_detected")
].copy()

print("Rules loaded:", len(resolved_rules))
print("Cycles resolved:", len(cycle_resolution_audit))
print("Residual cycles:", len(residual_cycles))
print(f"Saved cycle audit: {CYCLE_AUDIT_PATH}")

if not residual_cycles.empty:
    display(residual_cycles)
    raise ValueError(
        "Name-rule cycles were detected. Fix these before using the rules in PRL."
    )


Rules loaded: 956
Cycles resolved: 7
Residual cycles: 0
Saved cycle audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_reference_cycle_resolution_v1.csv


**What this code does:** Finds rules that point the same spelling to incompatible canonical forms and flags them for review.


In [6]:
# Cell 6 — Identify conflicting mappings

rule_conflicts = (
    resolved_rules.groupby(["name_type", "original_name_key"], dropna=False)
    .agg(
        canonical_options=("canonical_name_key", lambda values: " | ".join(sorted(set(values)))),
        canonical_option_count=("canonical_name_key", "nunique"),
        source_values=("original_name", lambda values: " | ".join(sorted(set(values)))),
    )
    .reset_index()
)

rule_conflicts = rule_conflicts[
    rule_conflicts["canonical_option_count"] > 1
].copy()

rule_conflicts.to_csv(CONFLICT_PATH, index=False)

print("Conflicting rule keys:", len(rule_conflicts))
display(rule_conflicts.head(20))


Conflicting rule keys: 0


,name_type,original_name_key,canonical_options,canonical_option_count,source_values


**What this code does:** Combines reviewed rules and frequency information into one reusable lookup table for names.


In [7]:
# Cell 7 — Build the reusable name-reference lookup table

# Rule-based entries
rule_lookup = resolved_rules[
    [
        "name_type",
        "original_name",
        "original_name_key",
        "direct_canonical_name",
        "direct_canonical_key",
        "canonical_name_key",
        "rule_source",
        "mapping_status",
        "mapping_path",
    ]
].copy()

rule_lookup["gender"] = pd.NA
rule_lookup["frequency"] = pd.NA
rule_lookup["rarity_category"] = pd.NA
rule_lookup["event_ids"] = pd.NA
rule_lookup["evidence_source"] = "cleaning_rule"

# Frequency entries
frequency_lookup = frequency.rename(
    columns={
        "name_original": "original_name",
        "name_key": "original_name_key",
        "freq": "frequency",
        "categorize": "rarity_category",
        "event_id": "event_ids",
    }
)[
    [
        "name_type",
        "original_name",
        "original_name_key",
        "gender",
        "frequency",
        "rarity_category",
        "event_ids",
    ]
].copy()

frequency_lookup["direct_canonical_name"] = frequency_lookup["original_name"]
frequency_lookup["direct_canonical_key"] = frequency_lookup["original_name_key"]
frequency_lookup["canonical_name_key"] = frequency_lookup["original_name_key"]
frequency_lookup["rule_source"] = pd.NA
frequency_lookup["mapping_status"] = "frequency_only"
frequency_lookup["mapping_path"] = frequency_lookup["original_name_key"]
frequency_lookup["evidence_source"] = "frequency_analysis"

lookup = pd.concat(
    [rule_lookup, frequency_lookup],
    ignore_index=True,
    sort=False,
)

lookup = lookup[
    [
        "name_type",
        "original_name",
        "original_name_key",
        "canonical_name_key",
        "gender",
        "frequency",
        "rarity_category",
        "event_ids",
        "evidence_source",
        "rule_source",
        "mapping_status",
        "mapping_path",
    ]
].sort_values(
    ["name_type", "original_name_key", "evidence_source"]
).reset_index(drop=True)

lookup.to_csv(LOOKUP_PATH, index=False)

print(f"Saved lookup: {LOOKUP_PATH}")
print("Lookup rows:", len(lookup))

lookup.head(20)


Saved lookup: C:\Users\samav\OneDrive\Desktop\sundondo\data\processed\name_reference_lookup_v1.csv
Lookup rows: 22982


,name_type,original_name,original_name_key,canonical_name_key,gender,frequency,rarity_category,event_ids,evidence_source,rule_source,mapping_status,mapping_path
0,first_name,abadesa,abadesa,abadesa,Female,1,Very rare,matrimonio-684,frequency_analysis,<NA>,frequency_only,abadesa
1,first_name,abadon,abadon,abadon,Male,1,Very rare,bautizo-3793,frequency_analysis,<NA>,frequency_only,abadon
2,first_name,abanuela,abanuela,abanuela,Female,1,Very rare,matrimonio-1011,frequency_analysis,<NA>,frequency_only,abanuela
3,first_name,abdon,abdon,abdon,Male,5,Common,bautizo-2720|matrimonio-586|matrimonio-741|ent...,frequency_analysis,<NA>,frequency_only,abdon
4,first_name,abel,abel,abel,Male,1,Very rare,entierro-37,frequency_analysis,<NA>,frequency_only,abel
5,first_name,abraham,abraham,abraham,Male,1,Very rare,entierro-1440,frequency_analysis,<NA>,frequency_only,abraham
6,first_name,acasio,acasio,acasio,Male,1,Very rare,entierro-123,frequency_analysis,<NA>,frequency_only,acasio
7,first_name,acceptacion,acceptacion,aceptacion,<NA>,<NA>,<NA>,<NA>,cleaning_rule,name_cleaning_rules.json,resolved,acceptacion -> aceptacion
8,first_name,acceptación,acceptacion,aceptacion,<NA>,<NA>,<NA>,<NA>,cleaning_rule,name_cleaning_rules.json,resolved,acceptacion -> aceptacion
9,first_name,acencia,acencia,acencia,Female,3,Common,bautizo-1099|matrimonio-23|matrimonio-89,frequency_analysis,<NA>,frequency_only,acencia


**What this code does:** Writes checks showing whether the rules, frequencies, and canonical keys are internally consistent.


In [8]:
# Cell 8 — Produce a validation report

validation = pd.DataFrame(
    [
        {"metric": "frequency_input_rows", "value": len(frequency)},
        {"metric": "frequency_first_names", "value": int((frequency["name_type"] == "first_name").sum())},
        {"metric": "frequency_last_names", "value": int((frequency["name_type"] == "last_name").sum())},
        {"metric": "first_name_abbreviation_rules", "value": len(first_name_abbreviations)},
        {"metric": "first_name_variant_rules", "value": len(first_name_variants)},
        {"metric": "last_name_variant_rules", "value": len(last_name_variants)},
        {"metric": "rule_mapping_cycles_resolved", "value": len(cycle_resolution_audit)},
        {"metric": "rule_mapping_residual_cycles", "value": len(residual_cycles)},
        {"metric": "rule_mapping_conflicts", "value": len(rule_conflicts)},
        {"metric": "final_lookup_rows", "value": len(lookup)},
        {"metric": "female_first_name_entries", "value": int(((frequency["name_type"] == "first_name") & (frequency["gender"] == "Female")).sum())},
        {"metric": "male_first_name_entries", "value": int(((frequency["name_type"] == "first_name") & (frequency["gender"] == "Male")).sum())},
        {"metric": "unknown_gender_first_name_entries", "value": int(((frequency["name_type"] == "first_name") & (frequency["gender"] == "Unknown")).sum())},
        {"metric": "very_rare_first_name_entries", "value": int(((frequency["name_type"] == "first_name") & (frequency["categorize"] == "Very rare")).sum())},
        {"metric": "most_common_first_name_entries", "value": int(((frequency["name_type"] == "first_name") & (frequency["categorize"] == "Most common")).sum())},
    ]
)

validation.to_csv(REFERENCE_VALIDATION_PATH, index=False)

display(validation)
print(f"Saved validation: {REFERENCE_VALIDATION_PATH}")
print(f"Saved conflicts: {CONFLICT_PATH}")


,metric,value
0,frequency_input_rows,22026
1,frequency_first_names,1830
2,frequency_last_names,2128
3,first_name_abbreviation_rules,20
4,first_name_variant_rules,409
5,last_name_variant_rules,503
6,rule_mapping_cycles_resolved,7
7,rule_mapping_residual_cycles,0
8,rule_mapping_conflicts,0
9,final_lookup_rows,22982


Saved validation: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_reference_validation_v1.csv
Saved conflicts: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_reference_rule_conflicts_v1.csv


## Persona enrichment

The next cells use the validated lookup to enrich persona_context.parquet. The enhanced context is saved separately, preserving the original context unchanged.


**What this code does:** Reads the existing persona context and confirms that the fields needed for enrichment are present.


In [9]:
# Cell 2 — Load inputs and validate the existing context schema

context = pd.read_parquet(CONTEXT_PATH)
lookup = pd.read_csv(LOOKUP_PATH, dtype=str)
overrides = pd.read_csv(PERSONA_OVERRIDE_PATH, dtype=str).fillna("")
gender_overrides = pd.read_csv(GENDER_OVERRIDE_PATH, dtype=str).fillna("")

required_override_columns = {
    "persona_idno", "linkage_full_name_clean", "linkage_first_name_clean",
    "linkage_lastname_clean", "override_reason", "reviewed_by", "reviewed_at",
}
missing_override_columns = required_override_columns - set(overrides.columns)
assert not missing_override_columns, f"Override file is missing: {sorted(missing_override_columns)}"
assert not overrides["persona_idno"].duplicated().any(), "Override file has duplicate persona IDs."
required_gender_override_columns = {"persona_idno", "reviewed_gender_evidence", "override_reason", "reviewed_by", "reviewed_at"}
missing_gender_override_columns = required_gender_override_columns - set(gender_overrides.columns)
assert not missing_gender_override_columns, f"Gender override file is missing: {sorted(missing_gender_override_columns)}"
assert not gender_overrides["persona_idno"].duplicated().any(), "Gender override file has duplicate persona IDs."
gender_overrides = gender_overrides.rename(columns={"override_reason": "gender_override_reason", "reviewed_by": "gender_reviewed_by", "reviewed_at": "gender_reviewed_at"})

context["persona_idno"] = context["persona_idno"].astype(str)

# Marriage-source godparents are genuine service-role appearances.  Preserve the
# source label, but make them eligible for linkage and the existing age safeguard.
context["role_standard_source"] = context["role_standard"]
marriage_godparent_mask = (
    context["event_type"].fillna("").astype(str).str.strip().str.lower().eq("marriage")
    & context["persona_type"].fillna("").astype(str).str.strip().str.lower().eq("godparent")
    & context["raw_role_clean"].fillna("").astype(str).str.strip().str.lower().eq("godparent")
)
context["role_standard_remapped"] = marriage_godparent_mask
context.loc[marriage_godparent_mask, "role_standard"] = "godparent"
context.loc[marriage_godparent_mask, "role_group"] = "service_role"
role_remap_audit = context.loc[marriage_godparent_mask, [
    "persona_idno", "event_key", "event_type", "persona_type",
    "raw_role_clean", "role_standard_source", "role_standard", "role_group",
]].copy()
assert role_remap_audit["role_standard"].eq("godparent").all(), "Marriage godparent remap failed."
print("Marriage godparent roles mapped for linkage:", len(role_remap_audit))
missing_override_personas = set(overrides["persona_idno"]) - set(context["persona_idno"])
assert not missing_override_personas, f"Overrides reference missing personas: {sorted(missing_override_personas)}"
missing_gender_override_personas = set(gender_overrides["persona_idno"]) - set(context["persona_idno"])
assert not missing_gender_override_personas, f"Gender overrides reference missing personas: {sorted(missing_gender_override_personas)}"

# Preserve all source values; overrides affect only the linkage-ready working context.
context["full_name_clean_source"] = context["full_name_clean"]
context["first_name_clean_source"] = context["first_name_clean"]
context["lastname_clean_source"] = context["lastname_clean"]
context = context.merge(overrides, on="persona_idno", how="left", validate="1:1")
context["name_override_applied"] = context["linkage_full_name_clean"].fillna("").str.strip().ne("")
for source_column, override_column in [
    ("full_name_clean", "linkage_full_name_clean"),
    ("first_name_clean", "linkage_first_name_clean"),
    ("lastname_clean", "linkage_lastname_clean"),
]:
    context[source_column] = context[override_column].where(
        context[override_column].fillna("").str.strip().ne(""), context[source_column]
    )

override_audit = context.loc[context["name_override_applied"], [
    "persona_idno", "full_name_clean_source", "full_name_clean",
    "override_reason", "reviewed_by", "reviewed_at",
]].copy()
print("Reviewed linkage-name overrides applied:", len(override_audit))

required_context_columns = {
    "persona_idno",
    "full_name_clean",
    "first_name_clean",
    "lastname_clean",
}

missing_context_columns = required_context_columns - set(context.columns)
assert not missing_context_columns, (
    f"persona_context is missing required columns: {sorted(missing_context_columns)}"
)

existing_gender_column = next(
    (column for column in ["gender_clean", "gender", "sex"] if column in context.columns),
    None,
)

print("Context rows:", len(context))
print("Lookup rows:", len(lookup))
print("Existing gender column:", existing_gender_column)


Marriage godparent roles mapped for linkage: 3260


Reviewed linkage-name overrides applied: 5
Context rows: 47072
Lookup rows: 22982
Existing gender column: gender


**What this code does:** Selects the relevant canonical-rule and frequency fields needed to enrich the personas.


In [10]:
rule_lookup = lookup[
    lookup["evidence_source"].eq("cleaning_rule")
].copy()

rule_lookup = rule_lookup[
    ["name_type", "original_name_key", "canonical_name_key", "mapping_path"]
].drop_duplicates()

rule_key_counts = rule_lookup.groupby(["name_type", "original_name_key"])["canonical_name_key"].nunique()
assert rule_key_counts.le(1).all(), "The validated rule lookup contains conflicting canonical targets."

print("Validated cleaning-rule rows:", len(rule_lookup))


Validated cleaning-rule rows: 938


**What this code does:** Builds frequency profiles and applies the reviewed Spanish C/S sound-equivalence only as comparison evidence.


In [11]:
# Cell 4 — Build canonical frequency profiles, including professor-approved Spanish C/S equivalence

frequency_lookup = lookup[
    lookup["evidence_source"].eq("frequency_analysis")
].copy()
frequency_lookup["frequency"] = pd.to_numeric(
    frequency_lookup["frequency"], errors="coerce"
)

# Preserve source spellings, but use a linkage-only signature for Spanish C/S before E or I.
# `sc` before E/I is collapsed first so dioniscio joins dionicio/dionisio.
def spanish_cs_ei_signature(value):
    value = normalize_name(value)
    value = re.sub(r"sc(?=[ei])", "s", value)
    return re.sub(r"c(?=[ei])", "s", value)

def summarize_gender(values):
    known = sorted({str(value).strip().lower() for value in values if pd.notna(value) and str(value).strip().lower() in {"male", "female"}})
    return known[0] if len(known) == 1 else ("ambiguous" if len(known) > 1 else "unknown")

first_rule_targets = (
    rule_lookup[rule_lookup["name_type"].eq("first_name")]
    .drop_duplicates("original_name_key")
    .set_index("original_name_key")["canonical_name_key"]
    .to_dict()
)
frequency_lookup["initial_canonical_key"] = frequency_lookup["original_name_key"].map(first_rule_targets).fillna(frequency_lookup["original_name_key"])

first_name_rows = frequency_lookup[frequency_lookup["name_type"].eq("first_name")].copy()
first_name_rows["spanish_cs_ei_signature"] = first_name_rows["initial_canonical_key"].map(spanish_cs_ei_signature)
cs_group_gender = first_name_rows.groupby("spanish_cs_ei_signature")["gender"].agg(summarize_gender).rename("group_gender")
cs_group_frequency = first_name_rows.groupby(["spanish_cs_ei_signature", "initial_canonical_key"], as_index=False)["frequency"].sum()
cs_group_frequency = cs_group_frequency.merge(cs_group_gender, on="spanish_cs_ei_signature", how="left")
eligible_cs_groups = cs_group_frequency.groupby("spanish_cs_ei_signature").filter(lambda group: group["initial_canonical_key"].nunique() > 1)
eligible_cs_groups = eligible_cs_groups[eligible_cs_groups["group_gender"].ne("ambiguous")].copy()
cs_targets = (
    eligible_cs_groups.sort_values(["spanish_cs_ei_signature", "frequency", "initial_canonical_key"], ascending=[True, False, True])
    .drop_duplicates("spanish_cs_ei_signature")
    .set_index("spanish_cs_ei_signature")["initial_canonical_key"]
    .to_dict()
)
first_name_phonetic_targets = {
    key: cs_targets.get(spanish_cs_ei_signature(key), key)
    for key in first_name_rows["initial_canonical_key"].dropna().unique()
}
first_name_rows["canonical_name_key"] = first_name_rows["initial_canonical_key"].map(first_name_phonetic_targets)
cs_rule_audit = first_name_rows[first_name_rows["initial_canonical_key"].ne(first_name_rows["canonical_name_key"])][[
    "original_name", "original_name_key", "initial_canonical_key", "canonical_name_key",
    "spanish_cs_ei_signature", "gender", "frequency",
]].drop_duplicates().sort_values(["canonical_name_key", "original_name_key"])

def summarize_gender(values):
    known = sorted({
        str(value).strip().lower()
        for value in values
        if pd.notna(value) and str(value).strip().lower() in {"male", "female"}
    })

    if len(known) == 1:
        return known[0]
    if len(known) > 1:
        return "ambiguous"
    return "unknown"


def categorize_aggregated_frequency(value):
    if pd.isna(value):
        return pd.NA
    if value == 1:
        return "Very rare"
    if value <= 2:
        return "Rare"
    if value <= 40:
        return "Common"
    return "Most common"

def build_frequency_profile(name_type):
    subset = frequency_lookup[
        frequency_lookup["name_type"].eq(name_type)
    ].copy()

    if name_type == "first_name":
        subset = first_name_rows.copy()
        profile_key = "canonical_name_key"
    else:
        subset["canonical_name_key"] = subset["original_name_key"]
        profile_key = "canonical_name_key"

    profile = (
        subset.groupby(profile_key, dropna=False)
        .agg(
            observed_frequency=("frequency", "sum"),
            gender_evidence=("gender", summarize_gender),
        )
        .reset_index()
    )
    profile = profile.rename(columns={profile_key: "original_name_key"})
    profile["rarity_category"] = profile["observed_frequency"].map(categorize_aggregated_frequency)

    return profile

first_name_frequency = build_frequency_profile("first_name")
last_name_frequency = build_frequency_profile("last_name")

print("First-name frequency profiles:", len(first_name_frequency))
print("Last-name frequency profiles:", len(last_name_frequency))
print("Professor-approved C/S first-name mappings applied:", len(cs_rule_audit))


First-name frequency profiles: 1576
Last-name frequency profiles: 2077
Professor-approved C/S first-name mappings applied: 94


**What this code does:** Adds canonical names, rarity categories, and gender evidence while preserving original values.


In [12]:
# Cell 5 — Add canonical names, rarity, and gender evidence

enriched = context.copy()

# Preserve the recorded place, while creating a conservative linkage-only key.
# This removes generic settlement descriptors (for example, 'pueblo de' and 'caserio')
# but never replaces the original event_place_clean value.
def place_linkage_key(value):
    value = normalize_name(value)
    value = re.sub(r'\b(?:pueblo|caserio|caserío|anexo|estancia|hacienda|paraje|distrito|comunidad)\b', ' ', value)
    value = re.sub(r'\bde\b', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

if 'event_place_clean' in enriched.columns:
    enriched['event_place_linkage_key'] = enriched['event_place_clean'].map(place_linkage_key)
else:
    enriched['event_place_linkage_key'] = ''
enriched['event_place_linkage_rule_applied'] = (
    enriched.get('event_place_clean', pd.Series('', index=enriched.index)).fillna('').astype(str).str.strip().ne('')
    & enriched['event_place_linkage_key'].ne(enriched.get('event_place_clean', pd.Series('', index=enriched.index)).fillna('').map(normalize_name))
)
place_linkage_audit = enriched.loc[enriched['event_place_linkage_rule_applied'], ['persona_idno', 'event_key', 'event_place_clean', 'event_place_linkage_key']].copy()
enriched["first_name_original_key"] = enriched["first_name_clean"].map(normalize_name)
enriched["lastname_original_key"] = enriched["lastname_clean"].map(normalize_name)

first_rules = rule_lookup[rule_lookup["name_type"].eq("first_name")].rename(
    columns={
        "original_name_key": "first_name_original_key",
        "canonical_name_key": "first_name_canonical_from_rule",
        "mapping_path": "first_name_mapping_path",
    }
)["first_name_original_key first_name_canonical_from_rule first_name_mapping_path".split()]

last_rules = rule_lookup[rule_lookup["name_type"].eq("last_name")].rename(
    columns={
        "original_name_key": "lastname_original_key",
        "canonical_name_key": "lastname_canonical_from_rule",
        "mapping_path": "lastname_mapping_path",
    }
)["lastname_original_key lastname_canonical_from_rule lastname_mapping_path".split()]

enriched = enriched.merge(first_rules, on="first_name_original_key", how="left", validate="m:1")
enriched = enriched.merge(last_rules, on="lastname_original_key", how="left", validate="m:1")

enriched["first_name_canonical_key"] = enriched["first_name_canonical_from_rule"].fillna(enriched["first_name_original_key"])
enriched["lastname_canonical_key"] = enriched["lastname_canonical_from_rule"].fillna(enriched["lastname_original_key"])
enriched["first_name_rule_applied"] = enriched["first_name_canonical_from_rule"].notna()
enriched["lastname_rule_applied"] = enriched["lastname_canonical_from_rule"].notna()

variant_targets = approved_variants.set_index("source_name_key")["canonical_name_key"].to_dict()
enriched["first_name_variant_canonical_key"] = enriched["first_name_original_key"].map(variant_targets)
enriched["first_name_variant_applied"] = enriched["first_name_variant_canonical_key"].notna()
assert (
    enriched.loc[enriched["first_name_variant_applied"], "first_name_canonical_key"]
    == enriched.loc[enriched["first_name_variant_applied"], "first_name_variant_canonical_key"]
).all(), "An approved name variant did not resolve to its specified canonical key."
variant_audit = enriched.loc[enriched["first_name_variant_applied"], [
    "persona_idno", "first_name_clean_source", "first_name_original_key",
    "first_name_canonical_key", "first_name_variant_canonical_key",
]].copy()

enriched["first_name_canonical_before_cs_rule"] = enriched["first_name_canonical_key"]
enriched["first_name_canonical_key"] = enriched["first_name_canonical_key"].map(first_name_phonetic_targets).fillna(enriched["first_name_canonical_key"])
enriched["first_name_cs_ei_rule_applied"] = enriched["first_name_canonical_key"].ne(enriched["first_name_canonical_before_cs_rule"])

first_profile = first_name_frequency.rename(columns={
    "original_name_key": "first_name_canonical_key",
    "observed_frequency": "first_name_frequency",
    "rarity_category": "first_name_rarity_category",
    "gender_evidence": "first_name_gender_evidence",
})
last_profile = last_name_frequency.rename(columns={
    "original_name_key": "lastname_canonical_key",
    "observed_frequency": "lastname_frequency",
    "rarity_category": "lastname_rarity_category",
})

enriched = enriched.merge(first_profile, on="first_name_canonical_key", how="left", validate="m:1")
enriched = enriched.merge(last_profile, on="lastname_canonical_key", how="left", validate="m:1")

def normalize_gender(value):
    value = normalize_name(value)
    if value in {"male", "masculino", "m", "hombre", "varon"}:
        return "male"
    if value in {"female", "femenino", "f", "mujer"}:
        return "female"
    return "unknown"

if existing_gender_column is None:
    enriched["existing_gender_evidence"] = "unknown"
else:
    enriched["existing_gender_evidence"] = enriched[existing_gender_column].map(normalize_gender)

enriched = enriched.merge(gender_overrides, on="persona_idno", how="left", validate="1:1")
enriched["gender_override_applied"] = enriched["reviewed_gender_evidence"].fillna("").str.strip().ne("")
enriched.loc[enriched["gender_override_applied"], "existing_gender_evidence"] = enriched.loc[enriched["gender_override_applied"], "reviewed_gender_evidence"].map(normalize_gender)
gender_override_audit = enriched.loc[enriched["gender_override_applied"], ["persona_idno", "full_name_clean_source", "event_key", "role_standard", "existing_gender_evidence", "gender_override_reason", "gender_reviewed_by", "gender_reviewed_at"]].copy()

enriched["first_name_gender_evidence"] = enriched["first_name_gender_evidence"].fillna("unknown")

def gender_status(row):
    existing = row["existing_gender_evidence"]
    inferred = row["first_name_gender_evidence"]
    if inferred not in {"male", "female"}:
        return "no_reliable_name_evidence"
    if existing == "unknown":
        return "name_evidence_only"
    if existing == inferred:
        return "agrees_with_existing_gender"
    return "conflicts_with_existing_gender"

enriched["gender_evidence_status"] = enriched.apply(gender_status, axis=1)

assert len(enriched) == len(context), "Enrichment changed the number of persona rows."
print("Enriched rows:", len(enriched))


Enriched rows: 47072


**What this code does:** Counts enrichment coverage and produces review rows for any gender or name-evidence concern.


In [13]:
# Cell 6 — Create auditable validation and review outputs

gender_conflicts = enriched[
    enriched["gender_evidence_status"].eq("conflicts_with_existing_gender")
].copy()

review_columns = [
    "persona_idno",
    "full_name_clean",
    "first_name_clean",
    "lastname_clean",
    "role_standard",
    "event_type",
    "existing_gender_evidence",
    "first_name_gender_evidence",
    "gender_evidence_status",
    "first_name_frequency",
    "first_name_rarity_category",
]
review_columns = [column for column in review_columns if column in gender_conflicts.columns]
gender_conflicts[review_columns].to_csv(GENDER_REVIEW_PATH, index=False)

validation = pd.DataFrame([
    {"metric": "input_persona_rows", "value": len(context)},
    {"metric": "output_persona_rows", "value": len(enriched)},
    {"metric": "row_count_preserved", "value": int(len(context) == len(enriched))},
    {"metric": "first_name_rules_applied", "value": int(enriched["first_name_rule_applied"].sum())},
    {"metric": "approved_name_variant_mappings", "value": len(approved_variants)},
    {"metric": "personas_with_approved_name_variant", "value": int(enriched["first_name_variant_applied"].sum())},
    {"metric": "personas_with_professor_approved_cs_rule", "value": int(enriched["first_name_cs_ei_rule_applied"].sum())},
    {"metric": "reviewed_gender_overrides_applied", "value": int(enriched["gender_override_applied"].sum())},
    {"metric": "marriage_godparent_roles_mapped_for_linkage", "value": int(enriched["role_standard_remapped"].sum())},
    {"metric": "lastname_rules_applied", "value": int(enriched["lastname_rule_applied"].sum())},
    {"metric": "personas_with_place_linkage_key", "value": int(enriched['event_place_linkage_key'].fillna('').str.strip().ne('').sum())},
    {"metric": "place_linkage_normalizations_applied", "value": int(enriched['event_place_linkage_rule_applied'].sum())},
    {"metric": "first_names_with_frequency_evidence", "value": int(enriched["first_name_frequency"].notna().sum())},
    {"metric": "lastnames_with_frequency_evidence", "value": int(enriched["lastname_frequency"].notna().sum())},
    {"metric": "name_gender_agreements", "value": int(enriched["gender_evidence_status"].eq("agrees_with_existing_gender").sum())},
    {"metric": "name_gender_conflicts_for_review", "value": len(gender_conflicts)},
    {"metric": "name_gender_evidence_only", "value": int(enriched["gender_evidence_status"].eq("name_evidence_only").sum())},
])

coverage = (
    enriched.groupby(["first_name_rarity_category", "gender_evidence_status"], dropna=False)
    .size()
    .reset_index(name="persona_count")
    .sort_values("persona_count", ascending=False)
)

display(validation)
display(coverage.head(20))
print("Gender-review rows:", len(gender_conflicts))


,metric,value
0,input_persona_rows,47072
1,output_persona_rows,47072
2,row_count_preserved,1
3,first_name_rules_applied,3456
4,approved_name_variant_mappings,34
5,personas_with_approved_name_variant,231
6,personas_with_professor_approved_cs_rule,1064
7,reviewed_gender_overrides_applied,3
8,marriage_godparent_roles_mapped_for_linkage,3260
9,lastname_rules_applied,7960


,first_name_rarity_category,gender_evidence_status,persona_count
3,Most common,agrees_with_existing_gender,33169
4,Most common,name_evidence_only,7502
0,Common,agrees_with_existing_gender,3059
1,Common,name_evidence_only,1978
9,Very rare,name_evidence_only,558
11,NaN,no_reliable_name_evidence,185
7,Rare,name_evidence_only,155
5,Most common,no_reliable_name_evidence,153
8,Very rare,agrees_with_existing_gender,140
6,Rare,agrees_with_existing_gender,106


Gender-review rows: 0


**What this code does:** Saves the enriched persona context and its validation reports as separate outputs; original context remains unchanged.


In [14]:
# Cell 7 — Save the separate enhanced context and audit reports

enriched.to_parquet(ENRICHED_CONTEXT_PATH, index=False)
validation.to_csv(CONTEXT_VALIDATION_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
override_audit.to_csv(OVERRIDE_AUDIT_PATH, index=False)
gender_override_audit.to_csv(GENDER_OVERRIDE_AUDIT_PATH, index=False)
variant_audit.to_csv(VARIANT_AUDIT_PATH, index=False)
cs_rule_audit.to_csv(CS_RULE_AUDIT_PATH, index=False)
place_linkage_audit.to_csv(PLACE_LINKAGE_AUDIT_PATH, index=False)
role_remap_audit.to_csv(ROLE_REMAP_AUDIT_PATH, index=False)

print(f"Saved enhanced context: {ENRICHED_CONTEXT_PATH}")
print(f"Saved validation: {CONTEXT_VALIDATION_PATH}")
print(f"Saved coverage: {COVERAGE_PATH}")
print(f"Saved gender review queue: {GENDER_REVIEW_PATH}")
print(f"Saved name-override audit: {OVERRIDE_AUDIT_PATH}")
print(f"Saved gender-override audit: {GENDER_OVERRIDE_AUDIT_PATH}")
print(f"Saved approved-variant audit: {VARIANT_AUDIT_PATH}")
print(f"Saved professor-approved C/S audit: {CS_RULE_AUDIT_PATH}")
print(f"Saved place-linkage audit: {PLACE_LINKAGE_AUDIT_PATH}")
print(f"Saved marriage-godparent role-remap audit: {ROLE_REMAP_AUDIT_PATH}")

sample_columns = [
    "persona_idno",
    "full_name_clean",
    "first_name_original_key",
    "first_name_canonical_key",
    "lastname_original_key",
    "lastname_canonical_key",
    "first_name_rarity_category",
    "first_name_gender_evidence",
    "gender_evidence_status",
]
display(enriched[[column for column in sample_columns if column in enriched.columns]].head(20))


Saved enhanced context: C:\Users\samav\OneDrive\Desktop\sundondo\data\interim\persona_context_name_enriched_v1.parquet
Saved validation: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_context_enrichment_validation_v1.csv
Saved coverage: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\name_context_enrichment_coverage_v1.csv
Saved gender review queue: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\review\name_reference_gender_conflicts_v1.csv
Saved name-override audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\persona_name_override_audit_v1.csv
Saved gender-override audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\persona_gender_override_audit_v1.csv
Saved approved-variant audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\approved_name_variant_application_audit_v1.csv
Saved professor-approved C/S audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\professor_approved_spanish_cs_rule_audit_v1.csv
Saved place-

,persona_idno,full_name_clean,first_name_original_key,first_name_canonical_key,lastname_original_key,lastname_canonical_key,first_name_rarity_category,first_name_gender_evidence,gender_evidence_status
0,persona-1,domingo ayquipa,domingo,domingo,ayquipa,aiquipa,Most common,male,agrees_with_existing_gender
1,persona-2,lucas ayquipa,lucas,lucas,ayquipa,aiquipa,Most common,male,agrees_with_existing_gender
2,persona-3,sevastiana quispe,sevastiana,sebastiana,quispe,quispe,Most common,female,agrees_with_existing_gender
3,persona-4,vicente guamani,vicente,vicente,guamani,huamani,Most common,male,agrees_with_existing_gender
4,persona-5,dominga lulia,dominga,dominga,lulia,lulia,Most common,female,agrees_with_existing_gender
5,persona-6,juan lulia,juan,juan,lulia,lulia,Most common,male,agrees_with_existing_gender
6,persona-7,jospha gomes,jospha,jospha,gomes,gomes,Very rare,female,name_evidence_only
7,persona-8,ignacio varientos,ignacio,ygnacio,varientos,barrientos,Most common,male,agrees_with_existing_gender
8,persona-9,bartola quispe,bartola,bartola,quispe,quispe,Most common,female,agrees_with_existing_gender
9,persona-10,jacinto quispe,jacinto,jacinto,quispe,quispe,Most common,male,agrees_with_existing_gender


---
## Marriage and birth-date quality audit

This prepares reliable date evidence for future sibling-candidate discovery. It preserves raw marriage and birth dates and does not create family links.


**What this code does:** Checks marriage and approximate birth-date information for later sibling discovery. It does not create links.


In [15]:
# Audit marriage dates and spouse birth dates for future PRL sibling discovery

from pathlib import Path
import pandas as pd
try:
    display
except NameError:
    def display(frame):
        print(frame.to_string(index=False))

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_MARRIAGES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'matrimonios_clean.csv'
DATE_AUDIT_PATH = PROJECT_ROOT / 'outputs' / 'reports' / 'marriage_birth_date_quality_audit_v1.csv'
BIRTH_ESTIMATE_REVIEW_PATH = PROJECT_ROOT / 'outputs' / 'review' / 'marriage_spouse_birth_estimate_review_v1.csv'
DATE_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
BIRTH_ESTIMATE_REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
assert RAW_MARRIAGES_PATH.exists(), f'Missing marriage input: {RAW_MARRIAGES_PATH}'

marriages = pd.read_csv(RAW_MARRIAGES_PATH, dtype=str).fillna('')
required_marriage_columns = {'identifier', 'event_date', 'husband_name', 'husband_lastname', 'husband_birth_date', 'wife_name', 'wife_lastname', 'wife_birth_date'}
missing_marriage_columns = required_marriage_columns - set(marriages.columns)
assert not missing_marriage_columns, f'Marriage input is missing: {sorted(missing_marriage_columns)}'
marriages['marriage_date_standardized'] = pd.to_datetime(marriages['event_date'], errors='coerce')

spouse_rows = []
for role in ['husband', 'wife']:
    part = marriages[['identifier', 'event_date', 'marriage_date_standardized', f'{role}_name', f'{role}_lastname', f'{role}_birth_date']].copy()
    part.columns = ['marriage_identifier', 'marriage_date_raw', 'marriage_date_standardized', 'first_name_raw', 'lastname_raw', 'birth_date_raw']
    part['spouse_role'] = role
    part['birth_date_standardized'] = pd.to_datetime(part['birth_date_raw'], errors='coerce')
    part['estimated_age_at_marriage_years'] = (part['marriage_date_standardized'] - part['birth_date_standardized']).dt.days / 365.2425
    part['birth_date_parse_status'] = pd.Series('missing', index=part.index)
    part.loc[part['birth_date_raw'].str.strip().ne('') & part['birth_date_standardized'].notna(), 'birth_date_parse_status'] = 'valid'
    part.loc[part['birth_date_raw'].str.strip().ne('') & part['birth_date_standardized'].isna(), 'birth_date_parse_status'] = 'invalid'
    part['eligible_for_future_sibling_search'] = (
        part['marriage_date_standardized'].notna()
        & part['birth_date_standardized'].notna()
        & part['estimated_age_at_marriage_years'].between(12, 80, inclusive='both')
    )
    spouse_rows.append(part)
spouse_dates = pd.concat(spouse_rows, ignore_index=True)

audit = pd.DataFrame([
    {'metric':'marriage_records','value':len(marriages)},
    {'metric':'marriage_dates_valid','value':int(marriages['marriage_date_standardized'].notna().sum())},
    {'metric':'spouse_rows','value':len(spouse_dates)},
    {'metric':'spouse_birth_dates_present','value':int(spouse_dates['birth_date_raw'].str.strip().ne('').sum())},
    {'metric':'spouse_birth_dates_valid','value':int(spouse_dates['birth_date_parse_status'].eq('valid').sum())},
    {'metric':'spouse_birth_dates_invalid','value':int(spouse_dates['birth_date_parse_status'].eq('invalid').sum())},
    {'metric':'plausible_marriage_age_estimates','value':int(spouse_dates['eligible_for_future_sibling_search'].sum())},
    {'metric':'automatic_family_links_created','value':0},
])
audit.to_csv(DATE_AUDIT_PATH, index=False)
review_columns = ['marriage_identifier', 'spouse_role', 'first_name_raw', 'lastname_raw', 'marriage_date_raw', 'marriage_date_standardized', 'birth_date_raw', 'birth_date_standardized', 'estimated_age_at_marriage_years', 'birth_date_parse_status', 'eligible_for_future_sibling_search']
spouse_dates[review_columns].to_csv(BIRTH_ESTIMATE_REVIEW_PATH, index=False)
display(audit)
display(spouse_dates.loc[spouse_dates['eligible_for_future_sibling_search'], review_columns].head(20))
print(f'Saved date-quality audit: {DATE_AUDIT_PATH}')
print(f'Saved spouse birth-estimate review file: {BIRTH_ESTIMATE_REVIEW_PATH}')
print('This output prepares review evidence only; it does not create PRL or family links.')


,metric,value
0,marriage_records,1719
1,marriage_dates_valid,1718
2,spouse_rows,3438
3,spouse_birth_dates_present,1200
4,spouse_birth_dates_valid,1200
5,spouse_birth_dates_invalid,0
6,plausible_marriage_age_estimates,1200
7,automatic_family_links_created,0


,marriage_identifier,spouse_role,first_name_raw,lastname_raw,marriage_date_raw,marriage_date_standardized,birth_date_raw,birth_date_standardized,estimated_age_at_marriage_years,birth_date_parse_status,eligible_for_future_sibling_search
509,M001,husband,jacinto,huamaní,1908-07-26,1908-07-26,1872-08-03,1872-08-03,35.976098,valid,True
510,M002,husband,leocadio,ramos,1908-07-26,1908-07-26,1880-08-01,1880-08-01,27.981410,valid,True
511,M003,husband,alejandro,urbano,1908-07-29,1908-07-29,1882-08-04,1882-08-04,25.982737,valid,True
512,M004,husband,isidro,quispe,1908-08-03,1908-08-03,1884-08-08,1884-08-08,23.984065,valid,True
513,M005,husband,hilario,bautista,1908-08-05,1908-08-05,1881-08-11,1881-08-11,26.982074,valid,True
514,M006,husband,sebastián,condori,1908-08-06,1908-08-06,1880-08-12,1880-08-12,27.981410,valid,True
515,M007,husband,antonio,huamani,1908-08-20,1908-08-20,1878-08-27,1878-08-27,29.980082,valid,True
516,M008,husband,mariano natividad,serrano,1908-08-23,1908-08-23,1890-08-27,1890-08-27,17.988049,valid,True
517,M009,husband,tomás domingo,cucho,1908-08-24,1908-08-24,1868-09-02,1868-09-02,39.973442,valid,True
518,M010,husband,celestino,serrano,1908-08-25,1908-08-25,1874-09-02,1874-09-02,33.977426,valid,True


Saved date-quality audit: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\reports\marriage_birth_date_quality_audit_v1.csv
Saved spouse birth-estimate review file: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\review\marriage_spouse_birth_estimate_review_v1.csv
This output prepares review evidence only; it does not create PRL or family links.


---
## Role-first gender enrichment for safe cluster membership

This creates a separate gender-enriched cluster membership file. The original `gender_clean` is preserved; role evidence is used first, and every disagreement is flagged for review. No identity cluster is changed.


**What this code does:** Uses clear role evidence to create a separate gender-enriched safe-membership file and flags disagreements.


In [16]:
# Create role-first linkage gender for the main safe cluster membership without overwriting source gender_clean

from pathlib import Path
import pandas as pd
try:
    display
except NameError:
    def display(frame):
        print(frame.to_string(index=False))

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SAFE_MEMBERSHIP_PATH = PROJECT_ROOT / 'data' / 'processed' / 'identity_cluster_membership_final_safe.parquet'
ENRICHED_CONTEXT_PATH = PROJECT_ROOT / 'data' / 'interim' / 'persona_context_name_enriched_v1.parquet'
GENDER_ENRICHED_MEMBERSHIP_PATH = PROJECT_ROOT / 'data' / 'processed' / 'identity_cluster_membership_gender_enriched_v1.parquet'
GENDER_ENRICHED_AUDIT_PATH = PROJECT_ROOT / 'outputs' / 'reports' / 'identity_cluster_gender_enrichment_audit_v1.csv'
GENDER_CONFLICT_REVIEW_PATH = PROJECT_ROOT / 'outputs' / 'review' / 'identity_cluster_gender_conflicts_v1.csv'
for path in [GENDER_ENRICHED_MEMBERSHIP_PATH.parent, GENDER_ENRICHED_AUDIT_PATH.parent, GENDER_CONFLICT_REVIEW_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)
assert SAFE_MEMBERSHIP_PATH.exists() and ENRICHED_CONTEXT_PATH.exists(), 'Run the base enrichment before gender enrichment.'

membership = pd.read_parquet(SAFE_MEMBERSHIP_PATH).copy()
context_gender = pd.read_parquet(ENRICHED_CONTEXT_PATH, columns=['persona_idno', 'existing_gender_evidence', 'first_name_gender_evidence']).copy()
membership['persona_idno'] = membership['persona_idno'].astype(str)
context_gender['persona_idno'] = context_gender['persona_idno'].astype(str)
assert membership['persona_idno'].is_unique and context_gender['persona_idno'].is_unique
gender_enriched = membership.merge(context_gender, on='persona_idno', how='left', validate='1:1')

male_roles = {'father', 'husband', 'husband_father', 'wife_father', 'godfather'}
female_roles = {'mother', 'wife', 'husband_mother', 'wife_mother', 'godmother'}
gender_enriched['role_standard'] = gender_enriched['role_standard'].fillna('').astype(str).str.strip().str.lower()
gender_enriched['role_gender_evidence'] = 'unknown'
gender_enriched.loc[gender_enriched['role_standard'].isin(male_roles), 'role_gender_evidence'] = 'male'
gender_enriched.loc[gender_enriched['role_standard'].isin(female_roles), 'role_gender_evidence'] = 'female'

for column in ['existing_gender_evidence', 'first_name_gender_evidence']:
    gender_enriched[column] = gender_enriched[column].fillna('unknown').astype(str).str.strip().str.lower()
    gender_enriched.loc[~gender_enriched[column].isin({'male', 'female'}), column] = 'unknown'

known_sources = gender_enriched[['role_gender_evidence', 'existing_gender_evidence', 'first_name_gender_evidence']]
gender_enriched['gender_evidence_conflict'] = known_sources.apply(lambda row: len({value for value in row if value in {'male', 'female'}}) > 1, axis=1)
gender_enriched['gender_linkage_resolved'] = gender_enriched['role_gender_evidence']
gender_enriched['gender_resolution_source'] = 'role'
needs_existing = gender_enriched['gender_linkage_resolved'].eq('unknown') & gender_enriched['existing_gender_evidence'].isin({'male', 'female'})
gender_enriched.loc[needs_existing, 'gender_linkage_resolved'] = gender_enriched.loc[needs_existing, 'existing_gender_evidence']
gender_enriched.loc[needs_existing, 'gender_resolution_source'] = 'existing_gender_evidence'
needs_name = gender_enriched['gender_linkage_resolved'].eq('unknown') & gender_enriched['first_name_gender_evidence'].isin({'male', 'female'})
gender_enriched.loc[needs_name, 'gender_linkage_resolved'] = gender_enriched.loc[needs_name, 'first_name_gender_evidence']
gender_enriched.loc[needs_name, 'gender_resolution_source'] = 'first_name_frequency_evidence'
gender_enriched.loc[gender_enriched['gender_linkage_resolved'].eq('unknown'), 'gender_resolution_source'] = 'unresolved'
gender_enriched['automatic_identity_cluster_change'] = False

assert len(gender_enriched) == len(membership), 'Gender enrichment changed membership row count.'
assert gender_enriched['gender_clean'].fillna('').astype(str).str.strip().eq(membership['gender_clean'].fillna('').astype(str).str.strip()).all(), 'Original gender_clean must remain unchanged.'
assert gender_enriched['automatic_identity_cluster_change'].eq(False).all()
gender_enriched.to_parquet(GENDER_ENRICHED_MEMBERSHIP_PATH, index=False)

audit = pd.DataFrame([
    {'metric':'safe_cluster_membership_rows','value':len(gender_enriched)},
    {'metric':'role_based_male','value':int(gender_enriched['role_gender_evidence'].eq('male').sum())},
    {'metric':'role_based_female','value':int(gender_enriched['role_gender_evidence'].eq('female').sum())},
    {'metric':'resolved_from_existing_gender','value':int(gender_enriched['gender_resolution_source'].eq('existing_gender_evidence').sum())},
    {'metric':'resolved_from_first_name_frequency','value':int(gender_enriched['gender_resolution_source'].eq('first_name_frequency_evidence').sum())},
    {'metric':'remaining_unknown_gender','value':int(gender_enriched['gender_linkage_resolved'].eq('unknown').sum())},
    {'metric':'gender_evidence_conflicts_for_review','value':int(gender_enriched['gender_evidence_conflict'].sum())},
    {'metric':'automatic_identity_cluster_changes','value':0},
])
audit.to_csv(GENDER_ENRICHED_AUDIT_PATH, index=False)
conflict_columns = ['persona_idno', 'final_cluster_id', 'final_historical_person_id', 'full_name_clean', 'role_standard', 'gender_clean', 'role_gender_evidence', 'existing_gender_evidence', 'first_name_gender_evidence', 'gender_linkage_resolved', 'gender_resolution_source']
gender_enriched.loc[gender_enriched['gender_evidence_conflict'], conflict_columns].to_csv(GENDER_CONFLICT_REVIEW_PATH, index=False)
display(audit)
print(f'Saved gender-enriched safe membership: {GENDER_ENRICHED_MEMBERSHIP_PATH}')
print(f'Saved gender conflict review: {GENDER_CONFLICT_REVIEW_PATH}')


,metric,value
0,safe_cluster_membership_rows,41251
1,role_based_male,12408
2,role_based_female,13434
3,resolved_from_existing_gender,12323
4,resolved_from_first_name_frequency,3006
5,remaining_unknown_gender,80
6,gender_evidence_conflicts_for_review,303
7,automatic_identity_cluster_changes,0


Saved gender-enriched safe membership: C:\Users\samav\OneDrive\Desktop\sundondo\data\processed\identity_cluster_membership_gender_enriched_v1.parquet
Saved gender conflict review: C:\Users\samav\OneDrive\Desktop\sundondo\outputs\review\identity_cluster_gender_conflicts_v1.csv
